# Analysis (Agent-based)
This notebook uses the aggregated output of `E1_data.ipynb`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from matplotlib.ticker import PercentFormatter
from rpy2.robjects import pandas2ri

from exp_utils import REPO_ROOT
from exp_plot_utils import (
    _configure_fonts,
    plot_emmeans_grouped_bar,
    GRAPH_COLOR,
    GRAPH_MAP,
    OKABE_ITO,
    SCENARIO_ORDER,
)

# exp_utils already puts REPO_ROOT on sys.path; r_utils lives in notebooks/.
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
from r_utils import (
    fit_lmer_full,
    fit_lmer_with_contrasts,
    build_contrast_summary,
    save_model_results,
    fit_family_direct_contrasts,
)

_configure_fonts()


In [ ]:
data_path = REPO_ROOT / "data" / "outputs" / "aggregated_expanded"
result_path = REPO_ROOT / "data" / "analysis" / "aggregated_expanded"
agg_path = result_path / "aggregated" / "agent"
cts_path = result_path / "contrasts"
for path in (agg_path, cts_path):
    path.mkdir(parents=True, exist_ok=True)

# pandas keeps the category dtypes and orders E1 wrote (run_scenario, graph_subtype, ...).
dft = pd.read_parquet(data_path / "agent_level_data.parquet")
dft.sample(10, random_state=0)


# Mixed-Effects Models / Aggregates

**Note on clustering structure**

For a given scenario, role assignments and adjacency matrices are fixed across statements. Specifically, for any given `(graph_type, seed)` pair, the network topology and role assignments are identical regardless of which statement is being discussed. This creates two non-independent dimensions in our data:

1. **Agents within the same `(graph_type, seed)` are not independent** — they share the same network structure and role assignments.
2. **The same agent-network configuration is reused across statements** — so observations from the same `(graph_type, seed)` pair but different statements are also not independent.

Errors should therefore be clustered by either `statement_id` or `(graph_type, seed)`, but not both simultaneously in a single-grouping estimator like `statsmodels` MixedLM. The right choice depends on which source of dependence is larger in practice, or we use crossed random effects `(1 | statement_id) + (1 | graph_type:seed)` via `pymer4`/`lme4`.

##### Installing `R` and its dependencies on macOS
1. Go to https://cran.r-project.org/bin/macosx/ and download the `R-4.5.x-arm64.pkg` installer. 
2. Run it: this installs `R` to `/Library/Frameworks/R.framework/` where `rpy2` expects it.
3. Run `Rscript -e 'install.packages(c("nloptr", "lme4", "lmerTest"), repos="https://cran.r-project.org")`
4. Install `rpy2` via `uv add rpy2`
#### Install additional packages
1. `Rscript -e 'install.packages("emmeans", repos="https://cran.r-project.org")'`
2. `Rscript -e 'install.packages("pbkrtest", repos="https://cran.r-project.org")'`
3. `Rscript -e 'install.packages("RLRsim", repos="https://cran.r-project.org")'`

## 1. Aggregated Stats

In [ ]:
COL_PRETTY_NAMES = {
    "agent_influence_proj_contrast": "Outgoing Influence",
    "agent_plasticity": "Plasticity",
    "agent_monotonicity": "Directedness",
}

In [ ]:
emm_by_col = {}  # reused by the decomposition grid below
for var_name in COL_PRETTY_NAMES:
    res = fit_lmer_full(
        df=dft,
        specification=f"""
            {var_name} ~ run_scenario * graph_family + 
            (1 | statement_id) + 
            (1 | graph_family:graph_seed) 
        """,
        emm_formula="~run_scenario * graph_family",
        reml=True,
    )
    res_pooled = fit_lmer_full(
        df=dft,
        specification=f"""
            {var_name} ~ run_scenario + 
            (1 | statement_id) + 
            (1 | graph_family:graph_seed)
        """,
        emm_formula="~run_scenario",
        reml=True,
    )
    emm_by_col[var_name] = res["emm"]
    save_model_results(res, f"{var_name}_scenario_family", agg_path)
    save_model_results(res_pooled, f"{var_name}_scenario", agg_path)
    print(f"Finished fitting model for {var_name} with run_scenario * graph_family")
    fig, ax, emm_plot = plot_emmeans_grouped_bar(
        emm_df=res["emm"],
        legend_title="Network Type",
        group_col="graph_family",
        group_colors=GRAPH_COLOR,
        group_labels=GRAPH_MAP,
        group_order=GRAPH_COLOR.keys(),
        y_label=f"{COL_PRETTY_NAMES.get(var_name, var_name)}",
        title="",
        figsize=(4.5, 2.75),
        show=True,
        out_fig=agg_path / f"{var_name}_grouped_bar.pdf",
    )

## 2. Influence (variance decomposition by scenario)

1. Do *some agents* systematically exert more influence than others, beyond their graph and
the statement being debated? The `agent_id:graph_id` ICC answers that, fitted separately
per scenario.
2. Does it hold beyond common drift, i.e. against non-neighbours of the same run?

In [ ]:
def fit_icc_with_lrt(
    df: pd.DataFrame,
    outcome: str,
    n_boot: int = 1000,
    parallel: str = "multicore",
    ncpus: int = 10,
    nsim_lrt: int = 10000,
) -> dict:
    """ICCs for agent_id:graph_id, graph_id and statement_id with parametric-bootstrap
    95% CIs, plus an exact restricted LRT for the agent_id:graph_id variance component."""
    with (ro.default_converter + pandas2ri.converter).context():
        ro.globalenv["df"] = ro.conversion.py2rpy(df)
        ro.globalenv["n_boot"] = n_boot
        ro.globalenv["boot_parallel"] = parallel
        ro.globalenv["boot_ncpus"] = ncpus
        ro.globalenv["nsim_lrt"] = nsim_lrt

        ro.r(
            """
            library(lme4)
            library(RLRsim)

            # Full model, carrying the agent random effect under test
            m_full <- lmer(OUTCOME ~ 1 + (1 | graph_id) + (1 | statement_id) + (1 | agent_id:graph_id),
                           data = df, REML = TRUE)

            var_comps <- function(fit) {
                vc <- as.data.frame(VarCorr(fit))
                c(agent = vc[vc$grp == "agent_id:graph_id", "vcov"],
                  graph = vc[vc$grp == "graph_id", "vcov"],
                  stmt  = vc[vc$grp == "statement_id", "vcov"],
                  resid = vc[vc$grp == "Residual", "vcov"])
            }
            icc_fn <- function(fit) {
                v <- var_comps(fit)
                setNames(v / sum(v), paste0("icc_", names(v)))
            }

            boot_result <- bootMer(m_full, FUN = icc_fn, nsim = n_boot,
                                   type = "parametric", use.u = FALSE,
                                   parallel = boot_parallel, ncpus = boot_ncpus)
            boot_ci <- apply(boot_result$t, 2, quantile, probs = c(0.025, 0.975), na.rm = TRUE)

            v <- var_comps(m_full)
            icc_df <- as.data.frame(as.list(c(icc_fn(m_full), setNames(v, paste0("var_", names(v))))))
            for (comp in colnames(boot_ci)) {
                icc_df[[paste0(comp, "_lo")]] <- boot_ci[1, comp]
                icc_df[[paste0(comp, "_hi")]] <- boot_ci[2, comp]
            }

            # Exact RLRT: reduced model drops the agent effect; m_alt carries only it
            m_reduced <- lmer(OUTCOME ~ 1 + (1 | graph_id) + (1 | statement_id), data = df, REML = TRUE)
            m_alt     <- lmer(OUTCOME ~ 1 + (1 | agent_id:graph_id), data = df, REML = TRUE)
            test <- exactRLRT(m = m_alt, mA = m_full, m0 = m_reduced, nsim = nsim_lrt)
            icc_df$lrt_stat <- as.numeric(test$statistic)
            icc_df$lrt_p    <- as.numeric(test$p.value)
        """.replace("OUTCOME", outcome)
        )

        return ro.conversion.rpy2py(ro.r("icc_df")).iloc[0].to_dict()


N_BOOT = 1000  # parametric bootstrap replicates for the ICC CIs
N_CPUS = 10
# outcome -> output file stem
ICC_OUTCOMES = {
    "agent_influence_proj_contrast": "influence_out_icc_proj_contrast",
}

icc_by_outcome = {}
for outcome, stem in ICC_OUTCOMES.items():
    icc_results = []
    for scenario in SCENARIO_ORDER:
        sub = dft[dft["run_scenario"] == scenario].dropna(subset=[outcome]).copy()
        sub["agent_id"] = sub["agent_id"].astype(
            str
        )  # agent_id is only unique within a graph
        # The agent variance needs each agent:graph observed in more than one statement.
        if sub["statement_id"].nunique() < 2:
            print(
                f"{outcome} | {scenario}: skipped, defined in {sub['statement_id'].nunique()} statement(s) only"
            )
            row = {}
        else:
            row = fit_icc_with_lrt(sub, outcome, n_boot=N_BOOT, ncpus=N_CPUS)
        row["run_scenario"] = scenario
        row["n_obs"] = len(sub)
        icc_results.append(row)
        if "icc_agent" in row:
            print(
                f"{outcome} | {scenario}: n={len(sub)}  icc_agent={row['icc_agent']:.3f}  icc_graph={row['icc_graph']:.3f}"
            )

    icc_df = pd.DataFrame(icc_results)
    icc_df.to_csv(agg_path / f"{stem}.csv", index=False)
    icc_by_outcome[outcome] = icc_df
    display(
        icc_df[
            [
                "run_scenario",
                "n_obs",
                "icc_agent",
                "icc_agent_lo",
                "icc_agent_hi",
                "icc_graph",
                "icc_stmt",
                "icc_resid",
                "lrt_p",
            ]
        ].round(4)
    )


In [ ]:
ICC_COMPONENTS = {"icc_agent": "Agent", "icc_graph": "Network"}
for outcome, stem in ICC_OUTCOMES.items():
    icc_df = icc_by_outcome[outcome]
    icc_long = pd.concat(
        [
            pd.DataFrame(
                {
                    "run_scenario": icc_df["run_scenario"],
                    "component": comp,
                    "emmean": icc_df[comp],
                    "lower.CL": icc_df[f"{comp}_lo"],
                    "upper.CL": icc_df[f"{comp}_hi"],
                }
            )
            for comp in ICC_COMPONENTS
        ]
    )

    fig, ax = plt.subplots(figsize=(4.5, 2.75))
    plot_emmeans_grouped_bar(
        emm_df=icc_long,
        group_col="component",
        group_order=list(ICC_COMPONENTS),
        group_colors={
            "icc_agent": OKABE_ITO["sky"],
            "icc_graph": OKABE_ITO["vermilion"],
        },
        group_labels=ICC_COMPONENTS,
        legend_title="",
        y_label="Intraclass correlation (%)",
        title=COL_PRETTY_NAMES.get(outcome, outcome),
        y_format=PercentFormatter(xmax=1.0, decimals=0),
        ax=ax,
    )
    ax.set_ylim(0, max(0.05, icc_long["upper.CL"].max() * 1.15))
    fig.tight_layout()
    fig.savefig(
        agg_path / f"{stem}_agent_graph_grouped_bar.pdf",
        bbox_inches="tight",
        format="pdf",
    )
    plt.show()


## 3. Contrasts

The four planned contrasts, computed marginally and within each graph family (plus the between-family interaction).
`fit_lmer_with_contrasts` and `build_contrast_summary` live in `notebooks/r_utils.py`, shared with E3.

In [ ]:
col_list = [
    "agent_plasticity",
    "agent_monotonicity",
    "agent_influence_proj_contrast",
]

AGENT_RANDOM_EFFECTS = (
    "(1 | statement_id) + (1 | run_scenario:statement_id) + (1 | graph_family:statement_id)"
    " + (1 | graph_id) + (1 | run_scenario:graph_id)"
    " + (1 | run_id) + (1 | agent_id:graph_id:run_scenario)"
)

results_dict = {}
for col in col_list:
    results_dict[col] = fit_lmer_with_contrasts(
        dft,
        outcome=col,
        random_effects=AGENT_RANDOM_EFFECTS,
        scenario_levels=SCENARIO_ORDER,
        spec_extra="* graph_family",
        group_var="graph_family",
        bootstrap=False,
    )
    print(f"contrasts done: {col}")

# Holm-adjusted p (p_holm) covers the family x contrast interaction tests only.
summary = build_contrast_summary(col_list, results_dict)

summary.to_csv(cts_path / "contrast_summary_agent.csv", index=False)
print(f"\n{len(summary)} contrast rows")
display(summary[summary.contrast_type == "interaction"].round(4))